In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load your text file
with open("source.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 2. Configure your text splitter
# You want a chunk_size around 500, and chunk_overlap around 50
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50) 

# 3. Create the chunks
chunks = splitter.create_documents([text_data])


# 4. Print the first 3 chunks to verify
for i in range(3):
    print(f"--- Chunk {i+1} ---")
    print(chunks[i].page_content)
    print("\n")

--- Chunk 1 ---
Quantum computing
A quantum computer is a (real or theoretical) computer that
exploits superposed and entangled states. Quantum computers
can be viewed as sampling from quantum systems. These
systems evolve in ways that operate on an enormous number
of possibilities simultaneously, though they remain subject to
strict computational constraints. By contrast, ordinary
("classical") computers operate according to deterministic
rules. (A classical computer can, in principle, be replicated by


--- Chunk 2 ---
a classical mechanical device, with only a simple multiple of
time cost. On the other hand (it is believed), a quantum
computer would require exponentially more time and energy
to be simulated classically.) It is widely believed that a
quantum computer could perform some calculations
exponentially faster than any classical computer. For
example, a large-scale quantum computer could break some
widely used public-key cryptographic schemes and aid


--- Chunk 3 ---
physic

## Embedding and storing in ChromaDB

In [2]:
import chromadb
from chromadb.utils import embedding_functions

# 5. Initialize ChromaDB (this creates a local folder to store the database)
chroma_client = chromadb.PersistentClient(path="./rag_database")

# 6. Set up the embedding model (downloads automatically on first run)
embedding_model = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# 7. Create a collection (think of this as a table in SQL)
collection = chroma_client.get_or_create_collection(
    name="knowledge_base",
    embedding_function=embedding_model
)

# 8. Extract the text strings and create unique IDs for each chunk
documents = [chunk.page_content for chunk in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

# 9. Insert into the database
collection.add(
    documents=documents,
    ids=ids
)

print(f"Successfully embedded and stored {collection.count()} chunks in ChromaDB!")

c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\phuc2\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\phuc2\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run 

Successfully embedded and stored 266 chunks in ChromaDB!


## Building the retriever


In [11]:
# 10. Write the retrieval function
def retrieve_documents(query, k=4):
    """
    Takes a user query, embeds it, and returns the top k matching chunks from ChromaDB.
    """
    results = collection.query(
        query_texts=[query], # Chroma automatically embeds this text for you
        n_results=k
    )
    
    # Chroma returns a dictionary with lists of ids, distances, and documents.
    # We just want the raw text documents for the top match.
    return results['documents'][0]

# 11. Test the retriever
test_query = "What is a qubit and how is it different from a classical bit?"
retrieved_chunks = retrieve_documents(test_query)

# 12. Display the results
print(f"\n--- Top 4 Results for: '{test_query}' ---")
for i, chunk in enumerate(retrieved_chunks):
    print(f"\nResult {i+1}:")
    print(chunk)


--- Top 4 Results for: 'What is a qubit and how is it different from a classical bit?' ---

Result 1:
Quantum information
Just as the bit is the basic concept of classical information theory, the qubit is the fundamental unit of
quantum information. The same term qubit is used to refer to an abstract mathematical model and to any
physical system that is represented by that model. A classical bit, by definition, exists in either of two
physical states, which can be denoted 0 and 1. A qubit is also described by a state, and two states, often
written
and

Result 2:
The basic unit of information in quantum computing is the
qubit (or "quantum bit"), which serves the same function as
the bit in ordinary, or "classical", computing.[1] However,
unlike a classical bit, which can be in one of two binary
Bloch sphere representation of a qubit.
states, a qubit can exist in a linear combination of two states,
The state
is a point on
known as a quantum superposition. When one measures a
the surface

## Step 2: The Guardrail Agent (Context Evaluator)

### build an LLM function that acts as a strict filter. It will look at the user's query, look at one chunk of text, and confidently state true (keep it) or false (discard it).

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load the variables from .env
load_dotenv()

# 2. Initialize the Gemini client
llm_client = genai.Client()

# 3. Define the Guardrail Agent for Gemini
def evaluate_chunk_relevance(query, chunk_text):
    """
    Uses Gemini to grade if a chunk is relevant to the query.
    Returns a boolean: True or False.
    """
    prompt = f"""You are a strict context evaluator grading document retrieval.
    Your job is to determine if the provided document chunk contains information relevant to answering the user's query.
    
    User Query: {query}
    Document Chunk: {chunk_text}
    
    Return a JSON object with a single boolean key "relevant". 
    Set it to true if the chunk contains useful information to answer the query, otherwise false.
    """
    
    # Call Gemini and force it to return strict JSON
    response = llm_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0 # Keep it strict and deterministic
        )
    )
    
    # Parse Gemini's text response into a Python dictionary
    result = json.loads(response.text)
    return result["relevant"]

# 4. Run the retrieved chunks through the Guardrail
print("\n--- Guardrail Agent Evaluation ---")
filtered_context = []

for i, chunk in enumerate(retrieved_chunks):
    is_relevant = evaluate_chunk_relevance(test_query, chunk)
    print(f"Chunk {i+1} Relevant? {is_relevant}")
    
    if is_relevant:
        filtered_context.append(chunk)

print(f"\n{len(filtered_context)} out of {len(retrieved_chunks)} chunks survived the filter.")


--- Guardrail Agent Evaluation ---


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

## Step 3: The Generator Agent

### This agent takes the filtered_context and the original query, and formulates a final, highly grounded answer.

* instruct the LLM to restrict its knowledge base only to the text we provide. If we don't do this, the LLM will fall back on its pre-trained weights, which defeats the entire purpose of RAG
* the Generator needs to know it's allowed to say "I don't know" rather than trying to guess if the Guardrail Agent filtered out all the chunks (leaving an empty list)

In [ ]:
# 16. Define the Generator Agent
def generate_answer(query, context_chunks):
    """
    Takes the user query and the filtered context chunks, and generates a grounded answer.
    """
    # If the guardrail filtered out everything, handle it gracefully
    if not context_chunks:
        return "I cannot answer this question based on the provided context."

    # Combine the surviving chunks into a single readable string
    context_text = "\n\n---\n\n".join(context_chunks)
    
    prompt = f"""You are an expert assistant. 
    Your task is to answer the user's query using strictly the provided Context. 
    If the Context does not contain the necessary information to answer the query, reply exactly with: 'I cannot answer this based on the provided context.' 
    Do not guess or use your own outside knowledge.
    
    Context:
    {context_text}
    
    User Query: {query}
    
    Answer:"""
    
    response = llm_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    return response.text

# 17. Run the Generator
print("\n--- Final Generated Answer ---")
final_answer = generate_answer(test_query, filtered_context)
print(final_answer)


--- Final Generated Answer ---
The qubit is the fundamental unit of quantum information, serving the same function as the bit in classical computing. It refers to an abstract mathematical model and any physical system represented by that model.

A classical bit exists in either of two physical states, denoted 0 and 1. In contrast, a qubit can exist in a linear combination of two states, known as a quantum superposition. If the coefficients describing a qubit's state are both nonzero, the qubit is in superposition; if either coefficient is zero, the qubit is effectively a classical bit. When a qubit is measured in the standard basis, the result is a classical bit.


In [ ]:
### CHANGE THE TEST QUERY TO SEE DIFFERENT RESULTS (check the failing logic by using a query that is likely not covered in the source.txt, like "What is the capital of France?")
#test_query = "What is the recommended cooking temperature for a medium-rare steak?"
final_answer = generate_answer(test_query, filtered_context)
print(final_answer)

I cannot answer this based on the provided context.


## Step 4: The Evaluator Agent (Factual Consistency Checker)

### Even when you provide good context, LLMs can still suffer from "attention drift" and sneak in outside knowledge.

Evaluator Agent acts as our final QA. It doesn't care about the user's original query. It only looks at two things:

* The Filtered Context we provided.

* The Generated Answer the system just wrote.

Its only job is to ask: Is every single claim in this answer explicitly backed up by the context?

In [ ]:
# 18. Define the Evaluator Agent (Hallucination Checker)
def evaluate_factuality(answer, context_chunks):
    """
    Evaluates if the generated answer is entirely supported by the provided context.
    Returns a boolean: True if factual, False if hallucinated.
    """
    # If there's no context, and the answer is our default "I cannot answer", it is factual.
    if not context_chunks:
         return True

    # Combine the surviving chunks into a single readable string
    context_text = "\n\n---\n\n".join(context_chunks)
    
    prompt = f"""You are a strict grading evaluator. 
    Your task is to check if the generated Answer is fully supported by the provided Context.
    If the Answer contains ANY information, claims, or facts not explicitly stated in the Context, it is a hallucination.
    
    Context:
    {context_text}
    
    Generated Answer: 
    {answer}
    
    Return a JSON object with a single boolean key "is_supported". 
    Set it to true if the answer is 100% faithful to the context, otherwise false.
    """
    
    response = llm_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0.0 # Strict grading
        )
    )
    
    result = json.loads(response.text)
    return result["is_supported"]

# 19. Test the Evaluator with the real answer
print("\n--- Factual Evaluation ---")
is_factual = evaluate_factuality(final_answer, filtered_context)
print(f"Is the generated answer fully supported by the context? {is_factual}")

# 20. Test the Evaluator with a deliberately hallucinated answer
fake_answer = "A qubit is the fundamental unit of quantum information. Unlike classical bits, qubits are powered by tiny microscopic lasers that spin at the speed of light."
print("\n--- Testing Hallucination Catch ---")
is_fake_caught = evaluate_factuality(fake_answer, filtered_context)
print(f"Is the fake answer supported by the context? {is_fake_caught}")


--- Factual Evaluation ---
Is the generated answer fully supported by the context? True

--- Testing Hallucination Catch ---
Is the fake answer supported by the context? False


## Step 5: The self correcting loop

Basically if the Evaluator Agent catches a hallucination answer, it blocks the output, appends a warning
to the Generator Agent, and forces it to rewrite the answer using strictly the filtered context

In [18]:
# 21. The Orchestrator Function
def run_agentic_rag(user_query, max_retries=3):
    print(f"\n🚀 Starting Agentic RAG for query: '{user_query}'")
    
    # --- PHASE 1: RETRIEVAL ---
    print("🔍 Retrieving documents from ChromaDB...")
    raw_chunks = retrieve_documents(user_query)
    
    # --- PHASE 2: GUARDRAIL (FILTERING) ---
    print("🛡️ Guardrail Agent evaluating context relevance...")
    filtered_chunks = []
    for chunk in raw_chunks:
        if evaluate_chunk_relevance(user_query, chunk):
            filtered_chunks.append(chunk)
            
    print(f"   -> {len(filtered_chunks)} out of {len(raw_chunks)} chunks passed the filter.")
    
    # Graceful exit if no context is relevant
    if not filtered_chunks:
        return "System: No relevant context found in the knowledge base. Cannot answer."

    # --- PHASE 3: THE SELF-CORRECTING LOOP ---
    attempt = 0
    feedback = ""
    
    while attempt < max_retries:
        attempt += 1
        print(f"\n Generator Agent drafting answer (Attempt {attempt})...")
        
        # If this is a retry, we inject a strict warning into the query
        current_query = user_query
        if feedback:
            print("   -> Injecting anti-hallucination feedback into prompt.")
            current_query = f"{user_query}\n\nWARNING: Your previous answer contained unverified facts. {feedback}"
        
        # Generate the draft
        draft_answer = generate_answer(current_query, filtered_chunks)
        
        # --- PHASE 4: EVALUATION ---
        print("Evaluator Agent checking facts against the source text...")
        is_factual = evaluate_factuality(draft_answer, filtered_chunks)
        
        if is_factual:
            print("Evaluator passed! The answer is 100% grounded.")
            return draft_answer
        else:
            print("Evaluator failed! Hallucination detected. Sending back to Generator...")
            # We provide specific feedback to the Generator for its next attempt
            feedback = "Please ensure every single claim is explicitly backed by the provided text. Do not add outside knowledge, even if it is technically true."
            
    # If we hit the max retries, we fail gracefully rather than lying to the user
    return "System: Failed to generate a fully grounded answer after maximum retries."

# 22. Test the full autonomous pipeline!
if __name__ == "__main__":
    test_question = "What is a qubit?"
    final_output = run_agentic_rag(test_question)
    
    print("\n==================================")
    print("FINAL DELIVERABLE TO USER:")
    print("==================================")
    print(final_output)


🚀 Starting Agentic RAG for query: 'What is a qubit?'
🔍 Retrieving documents from ChromaDB...
🛡️ Guardrail Agent evaluating context relevance...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.090942902s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},